In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

import pymedphys
from pymedphys._experimental.autosegmentation import filtering, indexing, mask, pipeline

In [ ]:
(
    _,
    _,
    _,
    ct_uid_to_structure_uid,
    structure_uid_to_ct_uids,
    _,
    structure_names_by_ct_uid,
    structure_names_by_structure_set_uid,
    _,
    _,
) = pipeline.get_dataset_metadata()

In [ ]:
structure_names = set()
for _, uid_structure_names in structure_names_by_structure_set_uid.items():
    for structure_name in uid_structure_names:
        structure_names.add(structure_name)

In [ ]:
structures_to_learn = ["patient", "brain", "eye_left", "eye_right"]
filters = {
    "study_set_must_have_all_of": structures_to_learn,
    "slice_at_least_one_of": ["brain", "eye_left", "eye_right"],
    "slice_must_have": ["patient"],
    "slice_cannot_have": [],
}

structure_uids, _ = pipeline.get_filtered_uids(filters)

In [ ]:
validation_structure_uids = structure_uids[2:5]
training_structure_uids = structure_uids[5::]

hold_out_structure_uids = structure_uids[0:2]

In [ ]:
assert len(set(validation_structure_uids).intersection(training_structure_uids)) == 0
assert len(set(hold_out_structure_uids).intersection(training_structure_uids)) == 0
assert len(set(validation_structure_uids).intersection(hold_out_structure_uids)) == 0

In [ ]:
_, validation_ct_uids = pipeline.get_filtered_uids(
    filters, structure_uids=validation_structure_uids
)
_, training_ct_uids = pipeline.get_filtered_uids(
    filters, structure_uids=training_structure_uids
)
_, hold_out_ct_uids = pipeline.get_filtered_uids(
    filters, structure_uids=hold_out_structure_uids
)

In [ ]:
assert len(set(validation_ct_uids).intersection(training_ct_uids)) == 0
assert len(set(hold_out_ct_uids).intersection(training_ct_uids)) == 0
assert len(set(validation_ct_uids).intersection(hold_out_ct_uids)) == 0

In [ ]:
mask_expansion = 5

validation_dataset = pipeline.create_dataset(
    validation_ct_uids, structures_to_learn, expansion=mask_expansion
)
training_dataset = pipeline.create_dataset(
    training_ct_uids, structures_to_learn, expansion=mask_expansion
)

hold_out_dataset = pipeline.create_dataset(
    hold_out_ct_uids, structures_to_learn, expansion=mask_expansion
)

In [ ]:
for ct_uid, x_grid, y_grid, input_array, output_array in validation_dataset.take(1):
    pass

In [ ]:
def diagnostic_plotting(x_grid, y_grid, input_array, output_array):
    plt.figure(figsize=(8, 6))

    x_grid = x_grid.numpy()
    y_grid = y_grid.numpy()
    input_array = input_array.numpy()[:, :, 0]
    output_array = output_array.numpy()

    for i in range(output_array.shape[-1]):
        contours = mask.get_contours_from_mask(x_grid, y_grid, output_array[:, :, i])
        for contour in contours:
            plt.plot(*contour.T)

    plt.axis("equal")
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    plt.pcolormesh(x_grid, y_grid, input_array, shading="nearest")
    plt.colorbar()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
diagnostic_plotting(x_grid, y_grid, input_array, output_array)

In [ ]:
sample_image = input_array
sample_mask = output_array

In [ ]:
def display(image, mask, prediction=None):
    diagnostic_plotting(x_grid, y_grid, image, mask)
    
    if prediction is None:
        try:
            prediction = model.predict(image[None, ...])[0, ...]
        except NameError:
            return
    
    diagnostic_plotting(x_grid, y_grid, image, prediction)
    
    
class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        display(sample_image, sample_mask)
        plt.show()
        print ('\nSample Prediction after epoch {}\n'.format(epoch+1))
    
    
display(sample_image, sample_mask)